# **实验三：基于 CNN 的手写数字识别**

使用 PyTorch 构建卷积神经网络，在 MNIST 数据集上完成手写数字（0-9）的分类任务。

# **0. 设置环境**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# **1. 数据准备**

MNIST 数据集包含 60,000 张训练图片和 10,000 张测试图片，每张为 28×28 的灰度手写数字。

## **1.1 设备检测**

检查并设置计算设备，确保优先利用 GPU 进行 CUDA 加速。

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## **1.2 数据预处理与加载**

定义预处理流程：先转换为张量，再进行官方推荐的标准化处理（均值 0.1307，标准差 0.3081）。

构建数据加载器，训练集开启打乱（shuffle）以提高模型的泛化能力。

In [3]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.68MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.06MB/s]

Training samples: 60000
Test samples: 10000


# **2. 模型构建**

构建卷积神经网络（CNN），结构为：
- **卷积层 1**：输入单通道灰度图，输出 32 个特征图，卷积核 3×3
- **最大池化层**：窗口 2×2，步长 2，用于特征下采样
- **卷积层 2**：进一步提取深层抽象特征，输出 64 个特征图
- **全连接隐藏层**：28×28 的图像经过两次 2×2 池化后尺寸变为 7×7，展平后映射到 128 维特征空间
- **输出层**：10 个节点对应数字 0-9 的分类概率

激活函数使用 ReLU。注意：PyTorch 的 CrossEntropyLoss 会自动做 Softmax，输出层不需要重复加。

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

实例化模型并将其部署到指定的计算设备上。

In [5]:
model = CNN().to(device)
print(model)

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)


# **3. 训练与评估**

## **3.1 损失函数与优化器**

采用经典的交叉熵损失函数和 Adam 优化器（学习率 0.001）。

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## **3.2 训练循环**

每个 epoch 的核心步骤：
1. 清空过往梯度（`optimizer.zero_grad()`）
2. 前向传播得到预测值（`model(data)`）
3. 计算当前批次的损失值（`criterion(outputs, target)`）
4. 反向传播计算梯度（`loss.backward()`）
5. 根据梯度更新网络参数（`optimizer.step()`）

In [7]:
def train(epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")

## **3.3 测试集评估**

评估阶段关闭梯度计算（`torch.no_grad()`），节省内存并加速推理。

In [8]:
def test():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = outputs.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    print(f"\n[Test] Accuracy on 10,000 test images: {100. * correct / total:.2f}%")

# **4. 运行实验**

5 轮迭代（epoch）足以让 MNIST 达到 98.5% 以上的优秀表现。

In [9]:
print("Starting PyTorch MNIST experiment...")
train(epochs=5)
test()

Starting PyTorch MNIST experiment...
Epoch [1/5] - Loss: 0.1294 - Accuracy: 95.99%
Epoch [2/5] - Loss: 0.0419 - Accuracy: 98.67%
Epoch [3/5] - Loss: 0.0282 - Accuracy: 99.11%
Epoch [4/5] - Loss: 0.0212 - Accuracy: 99.31%
Epoch [5/5] - Loss: 0.0157 - Accuracy: 99.50%

[Test] Accuracy on 10,000 test images: 98.95%
